# Domain 1 – Tyre Degradation: Degradation Curve Analysis

This notebook:
- Plots degradation curves per compound with linear fit overlay
- Compares linear vs polynomial fit quality (R²)
- Identifies fastest-degrading tracks
- Analyses compound cliffs (where deg accelerates)


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

from src.utils.paths import DOMAIN1_SILVER
print('Imports OK')

## 1. Load Silver Data

In [ ]:
clean_air = pd.read_parquet(DOMAIN1_SILVER / 'clean_air_laps.parquet')
stints = pd.read_parquet(DOMAIN1_SILVER / 'stints_degradation.parquet')

print(f'Clean-air laps: {len(clean_air):,}')
print(f'Stint records:  {len(stints):,}')
stints.head()

## 2. Degradation Curves per Compound (Scatter + Linear Fit Overlay)

In [ ]:
compound_col = 'tyre_compound' if 'tyre_compound' in clean_air.columns else 'Compound'
compounds = ['SOFT', 'MEDIUM', 'HARD']
compound_colors = {'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for ax, compound in zip(axes, compounds):
    subset = clean_air[
        (clean_air[compound_col] == compound) &
        (clean_air['stint_lap'] <= 40)
    ].copy()

    if subset.empty:
        ax.set_title(f'{compound} – No data')
        continue

    # Scatter
    ax.scatter(
        subset['stint_lap'], subset['LapTime'],
        alpha=0.1, color=compound_colors[compound], s=5, label='Laps'
    )

    # Median trend per stint_lap
    median_trend = subset.groupby('stint_lap')['LapTime'].median().reset_index()
    ax.plot(
        median_trend['stint_lap'], median_trend['LapTime'],
        color=compound_colors[compound], linewidth=2.5, label='Median pace'
    )

    # Linear fit overlay
    if len(median_trend) >= 3:
        m, b, _, _, _ = stats.linregress(median_trend['stint_lap'], median_trend['LapTime'])
        x_fit = np.linspace(median_trend['stint_lap'].min(), median_trend['stint_lap'].max(), 100)
        ax.plot(x_fit, m * x_fit + b, 'k--', linewidth=1.5,
                label=f'Linear fit\nSlope={m:.4f}s/lap')

    ax.set_title(f'{compound} Compound')
    ax.set_xlabel('Stint Lap')
    ax.set_ylabel('Lap Time (s)')
    ax.legend(fontsize=8)

plt.suptitle('Tyre Degradation Curves by Compound', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Linear vs Polynomial Fit Quality (R² Comparison)

In [ ]:
if 'r2_linear' in stints.columns and 'r2_poly' in stints.columns:
    r2_compare = stints[['compound', 'r2_linear', 'r2_poly']].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # R² by compound
    r2_melt = r2_compare.melt(
        id_vars='compound', value_vars=['r2_linear', 'r2_poly'],
        var_name='Model', value_name='R²'
    )
    r2_melt['Model'] = r2_melt['Model'].map({'r2_linear': 'Linear', 'r2_poly': 'Polynomial'})

    sns.boxplot(data=r2_melt, x='compound', y='R²', hue='Model', ax=axes[0])
    axes[0].set_title('R² by Compound: Linear vs Polynomial')
    axes[0].set_xlabel('Tyre Compound')

    # Scatter: linear R² vs polynomial R²
    axes[1].scatter(r2_compare['r2_linear'], r2_compare['r2_poly'],
                    alpha=0.4, s=20)
    axes[1].plot([0, 1], [0, 1], 'r--', label='Equal fit')
    axes[1].set_xlabel('R² Linear')
    axes[1].set_ylabel('R² Polynomial')
    axes[1].set_title('Linear R² vs Polynomial R² (per stint)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print('Mean R² by model and compound:')
    print(r2_compare.groupby('compound')[['r2_linear', 'r2_poly']].mean().round(3))
else:
    print('R² columns not yet available. Run domain1_degradation first.')

## 4. Fastest Degrading Tracks

In [ ]:
event_col = 'event' if 'event' in stints.columns else 'EventName'
if event_col in stints.columns and 'deg_rate_linear' in stints.columns:
    track_deg = (
        stints.groupby(event_col)['deg_rate_linear']
        .agg(['mean', 'median', 'std'])
        .reset_index()
        .sort_values('median', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.barh(
        track_deg[event_col], track_deg['median'],
        xerr=track_deg['std'],
        color=sns.color_palette('YlOrRd', len(track_deg)),
        capsize=3
    )
    ax.axvline(track_deg['median'].median(), color='blue', linestyle='--',
               label='Season median')
    ax.set_title('Median Linear Degradation Rate by Track (s/lap)')
    ax.set_xlabel('Degradation Rate (s/lap)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print('Top 5 fastest-degrading tracks:')
    print(track_deg.head())
else:
    print('event or deg_rate_linear columns not available.')

## 5. Compound Cliff Analysis

In [ ]:
compound_col_laps = 'tyre_compound' if 'tyre_compound' in clean_air.columns else 'Compound'
compounds = [c for c in ['SOFT', 'MEDIUM', 'HARD'] if c in clean_air[compound_col_laps].values]

fig, ax = plt.subplots(figsize=(14, 6))
compound_colors = {'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6'}

for compound in compounds:
    subset = clean_air[
        (clean_air[compound_col_laps] == compound) &
        (clean_air['stint_lap'].between(1, 40))
    ]
    if subset.empty:
        continue

    # Rolling lap-to-lap delta to find cliff
    trend = subset.groupby('stint_lap')['LapTime'].median().reset_index()
    trend['delta'] = trend['LapTime'].diff()  # lap-to-lap change

    ax.plot(
        trend['stint_lap'], trend['delta'].rolling(3, center=True).mean(),
        color=compound_colors.get(compound, 'grey'),
        linewidth=2, label=compound, marker='o', markersize=3
    )

ax.axhline(0, color='black', linestyle='--', alpha=0.4)
ax.set_title('Compound Cliff Analysis: Lap-to-Lap Pace Delta (3-lap rolling mean)')
ax.set_xlabel('Stint Lap')
ax.set_ylabel('Lap-to-Lap Delta (s)')
ax.legend(title='Compound')
plt.tight_layout()
plt.show()